# First-order and Newton Wasserstein flows for the energy distance

This notebook illustrates the first-order Wasserstein gradient flow and its undamped Newton lift for the squared MMD-type energy associated with the energy-distance kernel
\[
    k(x,y)=-\|x-y\|,
    \qquad
    f(\alpha)=\iint k(x,y)\,d(\alpha-\beta)(x)d(\alpha-\beta)(y).
\]
Equivalently,
\[
    f(\alpha)
    =2\mathbb E\|X-Y\|-\mathbb E\|X-X'\|-\mathbb E\|Y-Y'\|,
\]
where \(X,X'\sim\alpha\) and \(Y,Y'\sim\beta\).  The target \(\beta\) is a two-component Gaussian mixture, and both \(\alpha_t\) and \(\beta\) are represented by large empirical particle clouds.

The force is evaluated from the empirical sums.  Numerically, the singular distance is replaced by the tiny smooth approximation
\[
    \|x-y\|_\delta=\sqrt{\|x-y\|^2+\delta^2},
\]
only to avoid instabilities on coincident or nearly coincident particles.  The display is separated from the simulation: many particles drive the dynamics, KDE smoothing renders the density snapshots, and a farthest-point subset of particles is used for the trajectory panel.


In [1]:
from pathlib import Path
import shutil
import sys

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.collections import LineCollection
from matplotlib.colors import to_rgb
from scipy.ndimage import gaussian_filter

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks-figures":
    ROOT = ROOT.parent
sys.path.append(str(ROOT / "notebooks-figures"))

from figure_style import BLUE, RED, figure_dir, interp_color, remove_axes, save_pdf, setup_matplotlib

setup_matplotlib()
rng = np.random.default_rng(561)
out_dir = figure_dir("gradflow-second-order-momentum-mmd")
pde_dir = ROOT / "PDE4ML" / "figures" / out_dir.name
pde_dir.mkdir(parents=True, exist_ok=True)
thumb_dir = ROOT / "notebooks-figures" / "thumbnails"
thumb_dir.mkdir(parents=True, exist_ok=True)

# Remove obsolete panels from older versions of this figure.
for folder in (out_dir, pde_dir):
    for pdf in folder.glob("*.pdf"):
        pdf.unlink()


## Empirical source and target clouds

The source cloud is a compact single Gaussian on the left.  The target cloud is a larger empirical discretization of a two-Gaussian mixture.  The mild truncation removes visually irrelevant tail particles and makes the empirical evolution easier to read.

In [2]:
def sample_truncated_gaussian(mean, cov, n, radius=2.15):
    """Draw from a Gaussian conditioned to a Mahalanobis ball."""
    mean = np.asarray(mean, dtype=float)
    cov = np.asarray(cov, dtype=float)
    L = np.linalg.cholesky(cov)
    batches = []
    total = 0
    while total < n:
        z = rng.normal(size=(max(900, n), 2))
        z = z[np.sum(z * z, axis=1) <= radius**2]
        batches.append(mean + z @ L.T)
        total += len(z)
    return np.vstack(batches)[:n]


def sample_truncated_mixture(n, weights, means, covs, radius=2.25):
    counts = rng.multinomial(n, np.asarray(weights) / np.sum(weights))
    pieces = [
        sample_truncated_gaussian(mean, cov, count, radius=radius)
        for count, mean, cov in zip(counts, means, covs)
    ]
    return np.vstack(pieces)


n_particles = 1050
n_target = 1400

X0 = sample_truncated_gaussian(
    mean=[-0.62, 0.02],
    cov=[[0.017, 0.002], [0.002, 0.026]],
    n=n_particles,
    radius=2.08,
)

target_weights = np.array([0.55, 0.45])
target_means = np.array([[0.02, 0.31], [0.58, -0.25]])
target_covs = np.array([
    [[0.040, 0.010], [0.010, 0.030]],
    [[0.044, -0.010], [-0.010, 0.036]],
])
Ytarget = sample_truncated_mixture(
    n_target,
    target_weights,
    target_means,
    target_covs,
    radius=2.25,
)


## Energy-distance force

For the smoothed kernel \(k_\delta(x,y)=-\sqrt{\|x-y\|^2+\delta^2}\),
\[
    -2\nabla_x k_\delta(x,y)
    =2\frac{x-y}{\sqrt{\|x-y\|^2+\delta^2}}.
\]
Hence the velocity is the difference between two empirical averages of unit directions.  The clipping below is not part of the model; it only prevents a few particles from setting the scale of the visualization.

In [3]:
distance_smoothing = 0.028
force_scale = 0.66
velocity_clip = 0.62
speed_clip = 0.78


def mmd_velocity(X):
    """Exact empirical Wasserstein velocity for the smoothed energy-distance kernel."""
    diff_source = X[:, None, :] - X[None, :, :]
    radius_source = np.sqrt(np.sum(diff_source * diff_source, axis=2) + distance_smoothing**2)
    source_average = (diff_source / radius_source[:, :, None]).mean(axis=1)

    diff_target = X[:, None, :] - Ytarget[None, :, :]
    radius_target = np.sqrt(np.sum(diff_target * diff_target, axis=2) + distance_smoothing**2)
    target_average = (diff_target / radius_target[:, :, None]).mean(axis=1)

    V = 2.0 * force_scale * (source_average - target_average)
    norms = np.linalg.norm(V, axis=1)
    V *= np.minimum(1.0, velocity_clip / (norms + 1e-12))[:, None]
    return V


## First-order and Newton dynamics

The first row follows the overdamped gradient flow
\[
    \dot x_i=v_{\alpha_t}(x_i).
\]
The second row uses the infinite-momentum limit, i.e. the undamped Newton equation
\[
    \ddot x_i=v_{\alpha_t}(x_i)=-\nabla_W f(\alpha_t)(x_i),
\]
initialized with zero velocity.  This highlights the difference between reading the Wasserstein force as an instantaneous velocity and reading it as an acceleration.


In [4]:
def simulate_first_order(dt=0.040, n_steps=430, record_every=5):
    X = X0.copy()
    records = []
    for step in range(n_steps + 1):
        if step % record_every == 0:
            records.append(X.copy())
        X = X + dt * mmd_velocity(X)
    return np.asarray(records)


def simulate_newton(dt=0.018, n_steps=560, record_every=7):
    X = X0.copy()
    S = np.zeros_like(X)
    records = []
    for step in range(n_steps + 1):
        if step % record_every == 0:
            records.append(X.copy())
        S = S + dt * mmd_velocity(X)
        norms = np.linalg.norm(S, axis=1)
        S *= np.minimum(1.0, speed_clip / (norms + 1e-12))[:, None]
        X = X + dt * S
    return np.asarray(records)


records = {
    "first": simulate_first_order(),
    "newton": simulate_newton(),
}

# Representative display particles: trim only the outermost source rim, then run FPS.
center0 = X0.mean(axis=0)
C0_inv = np.linalg.inv(np.cov(X0.T))
mahal = np.einsum("ni,ij,nj->n", X0 - center0, C0_inv, X0 - center0)
candidates = np.flatnonzero(mahal <= np.quantile(mahal, 0.95))
P = X0[candidates]
chosen = [int(np.argmin(P[:, 0]))]
dist2 = np.sum((P - P[chosen[0]]) ** 2, axis=1)
for _ in range(1, 42):
    idx = int(np.argmax(dist2))
    chosen.append(idx)
    dist2 = np.minimum(dist2, np.sum((P - P[idx]) ** 2, axis=1))
traj_ids = candidates[np.asarray(chosen)]

summary = {
    key: {
        "shape": rec.shape,
        "final_mean": rec[-1].mean(axis=0),
        "final_cov_diag": np.diag(np.cov(rec[-1].T)),
    }
    for key, rec in records.items()
}
summary["target"] = {
    "mean": Ytarget.mean(axis=0),
    "cov_diag": np.diag(np.cov(Ytarget.T)),
}
summary


{'first': {'shape': (87, 1050, 2),
  'final_mean': array([0.26953912, 0.06160155]),
  'final_cov_diag': array([0.11156279, 0.10209238])},
 'newton': {'shape': (81, 1050, 2),
  'final_mean': array([ 0.43873405, -0.12433084]),
  'final_cov_diag': array([0.5378192 , 0.91054257])},
 'target': {'mean': array([0.27164227, 0.06358751]),
  'cov_diag': array([0.11090474, 0.10024194])}}

## Rendering helpers

A relatively broad KDE bandwidth is used for the snapshots, so that the panels show the mean-field density movement rather than the empirical sampling noise.  Dashed gray contours show the target KDE.

In [5]:
limit_clouds = [X0, Ytarget]
for rec in records.values():
    step = max(1, len(rec) // 32)
    limit_clouds.append(rec[::step].reshape(-1, 2))
all_points = np.vstack(limit_clouds)
# Newton trajectories can create a few far-away particles.  Use a robust crop so
# the density panels stay readable while preserving the visible inertial spread.
lo = np.quantile(all_points, 0.010, axis=0)
hi = np.quantile(all_points, 0.990, axis=0)
center = 0.5 * (lo + hi)
span = max(hi - lo)
pad = 0.075 * span
xlim = (center[0] - 0.5 * span - pad, center[0] + 0.5 * span + pad)
ylim = (center[1] - 0.5 * span - pad, center[1] + 0.5 * span + pad)

grid_n = 300
x_grid = np.linspace(*xlim, grid_n)
y_grid = np.linspace(*ylim, grid_n)
Xg, Yg = np.meshgrid(x_grid, y_grid)
pixel_size = (xlim[1] - xlim[0]) / (grid_n - 1)


def kde_density(points, bandwidth=0.095):
    H, _, _ = np.histogram2d(
        points[:, 0],
        points[:, 1],
        bins=grid_n,
        range=[xlim, ylim],
        density=True,
    )
    return gaussian_filter(H.T, sigma=bandwidth / pixel_size, mode="nearest")


def contour_levels(D):
    positive = D[D > np.quantile(D, 0.50)]
    return np.unique(np.quantile(positive, [0.55, 0.76, 0.91]))


def density_rgba(D, color, quantile=0.975):
    vmax = np.quantile(D[D > 0], quantile)
    value = np.clip(D / max(vmax, 1e-12), 0.0, 1.0)
    base = np.asarray(to_rgb(color))
    rgba = np.ones((*D.shape, 4))
    rgba[..., :3] = (1.0 - value[..., None]) + value[..., None] * base
    rgba[..., 3] = 0.74 * value
    return rgba


def add_target_contours(ax):
    ax.contour(
        Xg,
        Yg,
        target_density,
        levels=target_levels,
        colors=["#2d2d2d"],
        linewidths=0.38,
        alpha=0.28,
        linestyles="--",
        zorder=3,
    )


def add_colored_trajectory(ax, path, lw=0.48, alpha=0.47):
    segments = np.stack([path[:-1], path[1:]], axis=1)
    colors = [(*interp_color(t), alpha) for t in np.linspace(0, 1, len(segments))]
    ax.add_collection(LineCollection(segments, colors=colors, linewidths=lw, zorder=6))


def setup_panel(ax):
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal")
    ax.set_facecolor("white")
    remove_axes(ax)


target_density = kde_density(Ytarget, bandwidth=0.100)
target_levels = contour_levels(target_density)
snapshot_fracs = [0.00, 0.05, 0.14, 0.36, 1.00]
snapshot_tags = ["t000", "t005", "t014", "t036", "t100"]


## Exported PDF panels

The article assembles the panels in LaTeX and rotates the row labels there.  The generated PDFs contain no embedded titles.

In [6]:
def render_trajectory_panel(rec, filename):
    fig, ax = plt.subplots(figsize=(1.55, 1.36))
    setup_panel(ax)
    add_target_contours(ax)
    for idx in traj_ids:
        add_colored_trajectory(ax, rec[:, idx, :])
    ax.scatter(rec[0, traj_ids, 0], rec[0, traj_ids, 1], s=2.4, color=RED, edgecolor="none", zorder=7)
    ax.scatter(rec[-1, traj_ids, 0], rec[-1, traj_ids, 1], s=2.4, color=BLUE, edgecolor="none", zorder=8)
    fig.subplots_adjust(0, 0, 1, 1)
    save_pdf(fig, out_dir / filename, pad_inches=0.004)
    plt.close(fig)


def render_density_panel(points, color, filename):
    D = kde_density(points, bandwidth=0.095)
    fig, ax = plt.subplots(figsize=(1.55, 1.36))
    setup_panel(ax)
    ax.imshow(
        density_rgba(D, color),
        extent=[xlim[0], xlim[1], ylim[0], ylim[1]],
        origin="lower",
        interpolation="bilinear",
        zorder=1,
    )
    ax.contour(Xg, Yg, D, levels=contour_levels(D), colors=[color], linewidths=0.40, alpha=0.50, zorder=2)
    add_target_contours(ax)
    fig.subplots_adjust(0, 0, 1, 1)
    save_pdf(fig, out_dir / filename, pad_inches=0.004)
    plt.close(fig)


row_specs = {
    "first": "first",
    "newton": "newton",
}

for key, stem in row_specs.items():
    rec = records[key]
    render_trajectory_panel(rec, f"{stem}-trajectories.pdf")
    for frac, tag in zip(snapshot_fracs, snapshot_tags):
        snap_id = int(round(frac * (len(rec) - 1)))
        render_density_panel(rec[snap_id], interp_color(frac), f"{stem}-density-{tag}.pdf")

for pdf in out_dir.glob("*.pdf"):
    shutil.copy2(pdf, pde_dir / pdf.name)


## Contact-sheet thumbnail

The PNG below is only for the searchable figure-notebook gallery.  The paper uses the individual PDF panels exported above.

In [7]:
labels = ["GF", "Newton"]
keys = ["first", "newton"]
fig, axes = plt.subplots(2, 6, figsize=(8.2, 2.9), constrained_layout=False)
for row, (key, label) in enumerate(zip(keys, labels)):
    rec = records[key]
    ax = axes[row, 0]
    setup_panel(ax)
    add_target_contours(ax)
    for idx in traj_ids:
        add_colored_trajectory(ax, rec[:, idx, :], lw=0.42, alpha=0.43)
    ax.scatter(rec[0, traj_ids, 0], rec[0, traj_ids, 1], s=2.2, color=RED, edgecolor="none", zorder=7)
    ax.scatter(rec[-1, traj_ids, 0], rec[-1, traj_ids, 1], s=2.2, color=BLUE, edgecolor="none", zorder=8)
    ax.text(0.02, 0.98, label, transform=ax.transAxes, ha="left", va="top", fontsize=6.8, color="#242424")

    for col, (frac, tag) in enumerate(zip(snapshot_fracs, snapshot_tags), start=1):
        snap_id = int(round(frac * (len(rec) - 1)))
        D = kde_density(rec[snap_id], bandwidth=0.095)
        ax = axes[row, col]
        setup_panel(ax)
        color = interp_color(frac)
        ax.imshow(
            density_rgba(D, color),
            extent=[xlim[0], xlim[1], ylim[0], ylim[1]],
            origin="lower",
            interpolation="bilinear",
            zorder=1,
        )
        ax.contour(Xg, Yg, D, levels=contour_levels(D), colors=[color], linewidths=0.34, alpha=0.48, zorder=2)
        add_target_contours(ax)
        if row == 0:
            ax.text(0.03, 0.98, f"t={frac:.2g}", transform=ax.transAxes, ha="left", va="top", fontsize=6.5, color="#242424")

fig.subplots_adjust(left=0.002, right=0.998, top=0.998, bottom=0.002, wspace=0.018, hspace=0.018)
thumb_path = thumb_dir / "gradflow-second-order-momentum-mmd.png"
fig.savefig(thumb_path, dpi=220, bbox_inches="tight", pad_inches=0.010)
plt.close(fig)
thumb_path


PosixPath('/Users/gpeyre/Dropbox/github/ot4ml/notebooks-figures/thumbnails/gradflow-second-order-momentum-mmd.png')